In [1]:
%load_ext autoreload
%autoreload 2

In [8]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from jppype import vscode_theme

from fundus_vessels_toolkit.models.branch_digraph_gnn.data import BranchDigraphDataset
from fundus_vessels_toolkit.models.branch_digraph_gnn.losses import BranchDigraphMiner
from fundus_vessels_toolkit.models.branch_digraph_gnn.model import BranchDigraphModel
from fundus_vessels_toolkit.segment_to_graph.vbranch_digraph import VBranchDigraph
from train import DigraphGNNTrainer, DigraphGNNTrainerConfig

vscode_theme()


HTML(value="<style>\n        .cell-output-ipywidget-background {\n                background: transparent !imp…

In [4]:
import datetime

PATH = [
    Path("/run/media/gaby/GREY SSD/PostDoc/DATA/Fundus/" + folder)
    for folder in ["GAVE-train", "MAPLES-DR", "Fundus-AV", "LES-AV", "INSPIRE"]
]
RAW = [path / "1-images" for path in PATH]
AV = [path / "2-av-pred_CLEMENT" for path in PATH]
TOPO = [path / "3-topo" for path in PATH]
opts = dict(resize_to=1024, root="tmp/DATA2", ignore_recent=datetime.datetime(2026, 3, 8))
dataset = BranchDigraphDataset.load_from_dirs(RAW, TOPO, av_dir=AV, **opts)
train_set, val_set, test_set = dataset.split_loaders(train_ratio=0.7, val_ratio=0.15)

Found 284 branch digraphs...


Processing...
Done!
Preloading dataset: 100%|██████████| 284/284 [00:26<00:00, 10.92it/s]


## Visualize result from pred table


In [4]:
def parse_arborescence(preds, idx, opti=False):
    pred = preds.loc[idx] if isinstance(idx, str) else preds.iloc[idx]
    b_parent = np.fromstring(pred[("opti_" if opti else "") + "parent"], sep=",", dtype=int)
    b_dir = np.fromstring(pred[("opti_" if opti else "") + "dir"], sep=",", dtype=int)
    b_av = np.fromstring(pred["av"], sep=",", dtype=int)
    return pred.name, b_parent, b_dir, b_av


### Load model from checkpoint


In [9]:
model = (
    DigraphGNNTrainer.load_from_checkpoint("../../lightning_logs/4n10j81v/checkpoints/epoch=79-step=2640.ckpt")
    .model.cuda()
    .eval()
)

In [16]:
ID = 45
with torch.inference_mode():
    out: BranchDigraphModel.Output = model(test_set.get(ID).cuda())
pred_parent, pred_dir = out.max_parent(), out.dir_logit > 0
pred_parent, pred_dir, pred_av = out.optimal_tree

gt_digraph, _, od_yx, _ = test_set.get_sample(ID)
assert VBranchDigraph.has_fp_av_p(gt_digraph)
valid_branch = ~gt_digraph.branch_fp()
av_gt = gt_digraph.branch_av_class() <= 1
print(((out.av_logit.numpy(force=True)[valid_branch] > 0) == av_gt[valid_branch]).mean())
print(((pred_av.numpy(force=True)[valid_branch] > 0) == av_gt[valid_branch]).mean())


0.9080882352941176
0.9264705882352942


In [ ]:
from pytorch_metric_learning import losses, distances


pairs = BranchDigraphMiner(triplet=False)(out)
triplet = BranchDigraphMiner(triplet=True)(out)

CosSim = distances.CosineSimilarity()
(
    losses.ContrastiveLoss(distance=CosSim, pos_margin=1, neg_margin=0)(
        out.b1_embedding_gt_tail_tip(), indices_tuple=pairs
    ),
    losses.TripletMarginLoss()(out.b1_embedding_gt_tail_tip(), indices_tuple=triplet),
)

(tensor(1.1584, device='cuda:0'), tensor(1.1024, device='cuda:0'))

In [21]:
m, pred_tree = test_set.show_tree_diff(
    out.name,
    pred_parent.numpy(force=True),
    pred_dir.numpy(force=True),
    out.fp_logit.numpy(force=True) > 0,
    pred_av.numpy(force=True) > 0,
)
m

[ WARN:0@258.622] global loadsave.cpp:1617 imencodeWithMetadata Unsupported depth image for selected encoder is fallbacked to CV_8U.


GridBox(children=(HTML(value='<h3 style="text-align: center;">GT Tree: image24</h3>'), HTML(value='<h3 style="…

In [9]:
m.views[0].goto(pred_tree.branch(177).midpoint().xy, scale=3)

In [10]:
out.av_logit[211]

tensor(-5.6672, device='cuda:0')

In [11]:
B_ID = 178
pred_av[B_ID], out.av_logit[B_ID], pred_parent[B_ID], out.max_parent()[B_ID]

(tensor(-5.0150, device='cuda:0'),
 tensor(-2.5944, device='cuda:0'),
 tensor(115, device='cuda:0'),
 tensor(115, device='cuda:0'))

In [27]:
np.set_printoptions(linewidth=200, precision=2, suppress=True)
d = out.to_digraph()
d.lines_info(b1=101, sort_by_p=True).round(3).head(20)

,b0,b1,tip0,tip1,line_p,av_p,total_p,b0_dir_p,b1_dir_p
0,99,101,1,0,0.241,0.345,1.211,0.962,0.978
1,115,101,1,0,0.467,0.005,1.152,0.393,0.978
2,114,101,0,0,0.179,0.002,1.092,0.849,0.978
3,260,101,1,0,0.015,0.558,0.999,0.990,0.978
4,-1,101,0,0,0.018,0.000,0.996,0.999,0.978
5,90,101,1,0,0.007,0.560,0.989,0.986,0.978
6,104,101,1,0,0.004,0.542,0.989,0.991,0.978
7,82,101,0,0,0.000,0.540,0.982,0.985,0.978
8,239,101,0,0,0.000,0.546,0.975,0.972,0.978
9,83,101,0,0,0.011,0.490,0.828,0.656,0.978


In [22]:
import tqdm

from fundus_toolkits import AVLabel
from fundus_toolkits.utils.geometric import Point
from fundus_vessels_toolkit.segment_to_graph.av_tree_parsing import naive_infer_arborescence

name = []
pred_parent_acc = []
pred_dir_acc = []
pred_av_acc = []
opti_parent_acc = []
opti_dir_acc = []
opti_av_acc = []
baseline_parent_acc = []
baseline_dir_acc = []
baseline_av_acc = []
node_ratio = []
branch_ratio = []

eval_set = test_set

with torch.inference_mode():
    for i in tqdm.tqdm(range(len(eval_set))):
        gt_digraph, _, od_yx, _ = eval_set.get_sample(i)
        assert gt_digraph.branch_dir_p is not None
        assert gt_digraph.graph is not None, "Graph must be loaded to infer tree"

        node_ratio += [gt_digraph.graph.node_count / eval_set.graphs[i].node_count]
        branch_ratio += [gt_digraph.graph.branch_count / eval_set.graphs[i].branch_count]

        valid_branch = ~gt_digraph.branch_fp()
        av_gt = (gt_digraph.branch_av_class() <= 1)[valid_branch]
        od = Point.parse(od_yx)

        art_branch = gt_digraph.graph.branch_attr["av"] == AVLabel.ART
        vei_branch = gt_digraph.graph.branch_attr["av"] == AVLabel.VEI
        parent_base = -np.ones(gt_digraph.graph.branch_count, dtype=np.int_)
        dir_base = np.zeros(gt_digraph.graph.branch_count, dtype=np.bool_)
        parent_base[art_branch], dir_base[art_branch] = naive_infer_arborescence(
            gt_digraph.graph, od, branch_subset=art_branch
        )
        parent_base[vei_branch], dir_base[vei_branch] = naive_infer_arborescence(
            gt_digraph.graph, od, branch_subset=vei_branch
        )
        parent_base, dir_base = parent_base[valid_branch], dir_base[valid_branch]
        baseline_parent_acc.append((parent_base == gt_digraph.max_parent()[valid_branch]).mean())
        baseline_dir_acc.append((dir_base == (gt_digraph.branch_dir_p[valid_branch] > 0.5)).mean())
        baseline_av_acc.append((art_branch[valid_branch] == av_gt).mean())

        out = model(eval_set.get(i).cuda())
        name += [out.name]
        pred_parent, pred_dir = out.max_parent(), out.dir_logit > 0

        pred_parent, pred_dir = pred_parent.numpy(force=True), pred_dir.numpy(force=True)
        pred_parent, pred_dir = pred_parent[valid_branch], pred_dir[valid_branch]
        av_logit = out.av_logit.numpy(force=True)[valid_branch]
        pred_parent_acc.append((pred_parent == gt_digraph.max_parent()[valid_branch]).mean())
        pred_dir_acc.append((pred_dir == (gt_digraph.branch_dir_p[valid_branch] > 0.5)).mean())
        pred_av_acc.append(((av_logit > 0) == av_gt).mean())

        opti_parent, opti_dir, opti_av = out.optimal_tree
        opti_parent, opti_dir = opti_parent.numpy(force=True), opti_dir.numpy(force=True)
        opti_parent, opti_dir = opti_parent[valid_branch], opti_dir[valid_branch]
        opti_parent_acc.append((opti_parent == gt_digraph.max_parent()[valid_branch]).mean())
        opti_dir_acc.append((opti_dir == (gt_digraph.branch_dir_p[valid_branch] > 0.5)).mean())

        opti_av = opti_av.numpy(force=True)[valid_branch]
        opti_av_acc.append(((opti_av > 0) == av_gt).mean())

100%|██████████| 46/46 [00:12<00:00,  3.62it/s]


In [24]:
np.mean(pred_parent_acc), np.mean(opti_parent_acc), np.mean(baseline_parent_acc)

(np.float64(0.9020000720203407),
 np.float64(0.9004582417332795),
 np.float64(0.8407415480698803))

In [25]:
np.mean(pred_dir_acc), np.mean(opti_dir_acc), np.mean(baseline_dir_acc)

(np.float64(0.9915838634113934),
 np.float64(0.9914613647940189),
 np.float64(0.9406805112341713))

In [26]:
np.mean(pred_av_acc), np.mean(opti_av_acc), np.mean(baseline_av_acc)

(np.float64(0.9353869148721345),
 np.float64(0.9074122274298257),
 np.float64(0.9484115983934909))

In [28]:
(np.array(pred_parent_acc) - np.array(pred_parent_acc)).argsort()[::-1]

array([45, 44, 43, 42, 41, 40, 39, 38, 37, 36, 35, 34, 33, 32, 31, 30, 29,
       28, 27, 26, 25, 24, 23, 22, 21, 20, 19, 18, 17, 16, 15, 14, 13, 12,
       11, 10,  9,  8,  7,  6,  5,  4,  3,  2,  1,  0])